# STRATZ Dota Analytics API Examples

Run the cells in order after `docker compose up -d --build`. The notebook calls the local API only and never reads the STRATZ token.

In [1]:
from __future__ import annotations

import time
from typing import Any

import httpx
import pandas as pd
from IPython.display import JSON, Markdown, display

API_BASE = "http://127.0.0.1:8000"
ACCOUNT_ID = 175966938
client = httpx.Client(timeout=90, trust_env=False)

In [2]:
def api_request(method: str, path: str, *, params: dict[str, Any] | None = None, allow_error: bool = False) -> tuple[int, Any]:
    response = client.request(method, f"{API_BASE}{path}", params=params)
    try:
        payload = response.json()
    except ValueError:
        payload = response.text
    if not allow_error:
        response.raise_for_status()
    return response.status_code, payload


def show(title: str, payload: Any) -> None:
    display(Markdown(f"## {title}"))
    if isinstance(payload, (dict, list)):
        display(JSON(payload, expanded=False))
    else:
        display(Markdown(f"```text\n{payload}\n```"))

## Health

In [3]:
status, health = api_request("GET", "/health")
show(f"GET /health -> {status}", health)

## GET /health -> 200

<IPython.core.display.JSON object>

## Refresh Player Data

A refresh may return `429` while the 15-minute cooldown is active.

In [4]:
refresh_path = f"/v1/players/{ACCOUNT_ID}/refresh"
status, refresh = api_request("POST", refresh_path, allow_error=True)
show(f"POST {refresh_path} -> {status}", refresh)
job_id = refresh.get("id") if status == 202 and isinstance(refresh, dict) else None

## POST /v1/players/175966938/refresh -> 429

<IPython.core.display.JSON object>

In [5]:
if job_id:
    for _ in range(30):
        status, job = api_request("GET", f"/v1/jobs/{job_id}")
        if job["status"] in {"done", "failed"}:
            break
        time.sleep(2)
    show(f"GET /v1/jobs/{job_id} -> {status}", job)
else:
    display(Markdown("No new refresh job was created. Continue with cached player data."))

No new refresh job was created. Continue with cached player data.

## Player Analytics

In [6]:
requests_to_run = {
    "summary": (f"/v1/players/{ACCOUNT_ID}/summary", {"limit": 100}),
    "breakdowns": (f"/v1/players/{ACCOUNT_ID}/breakdowns", {"limit": 100}),
    "matches": (f"/v1/players/{ACCOUNT_ID}/matches", {"limit": 20}),
    "heroes": (f"/v1/players/{ACCOUNT_ID}/heroes", {"limit": 100}),
    "achievements": (f"/v1/players/{ACCOUNT_ID}/achievements", {"limit": 100}),
    "rating": (f"/v1/players/{ACCOUNT_ID}/rating", {"limit": 100}),
}

responses: dict[str, Any] = {}
for name, (path, params) in requests_to_run.items():
    status, payload = api_request("GET", path, params=params, allow_error=True)
    responses[name] = payload
    print(f"{name}: {status}")

summary: 200
breakdowns: 200
matches: 200
heroes: 200
achievements: 200
rating: 200


In [7]:
analysis_errors = {
    name: payload["detail"]
    for name, payload in responses.items()
    if isinstance(payload, dict) and payload.get("detail")
}

if analysis_errors:
    display(Markdown("### Player data is not ready"))
    display(pd.DataFrame(analysis_errors.items(), columns=["endpoint", "detail"]))
    display(Markdown("Run the refresh cells above and wait for a job with status `done`. If it fails with `STRATZ token was rejected`, replace `STRATZ_TOKEN` in `.env` and run `docker compose up -d --force-recreate api`."))
else:
    show("Player summary", responses["summary"])
    matches = pd.DataFrame(responses["matches"])
    display(Markdown("### Recent matches"))
    display(matches.head(20))
    heroes = pd.DataFrame(responses["heroes"])
    display(Markdown("### Player heroes"))
    display(heroes.sort_values(["games", "winRate"], ascending=[False, False]).head(30))

## Player summary

<IPython.core.display.JSON object>

### Recent matches

,matchId,startedAt,durationSeconds,heroId,gameMode,gameModeName,lobbyType,rankedBucket,rankedLabel,won,kills,deaths,assists,gpm,xpm,heroDamage,towerDamage,heroHealing,lastHits,laneRole
0,8888824424,2026-07-09T19:27:50Z,1737,14,23,Turbo,0,unranked,Unranked,False,6,5,6,995,1746,23311,717,0,180,None
1,8888227610,2026-07-09T12:38:51Z,1460,3,23,Turbo,0,unranked,Unranked,False,3,8,7,594,1270,16235,107,0,39,None
2,8888187282,2026-07-09T12:10:11Z,1381,126,23,Turbo,0,unranked,Unranked,False,3,7,5,744,1574,7763,233,0,79,None
3,8886131605,2026-07-07T21:28:23Z,2483,60,22,Ranked All Pick,7,ranked,Ranked,False,1,13,11,336,581,11064,26,0,120,None
4,8886077137,2026-07-07T20:34:47Z,2579,84,22,Ranked All Pick,7,ranked,Ranked,True,12,12,17,650,762,23528,2944,0,189,None
5,8885808627,2026-07-07T17:03:21Z,1895,63,23,Turbo,0,unranked,Unranked,True,14,13,17,1807,2731,84610,11998,0,230,None
6,8885724269,2026-07-07T16:00:22Z,1191,99,23,Turbo,0,unranked,Unranked,True,3,2,11,1136,1476,13224,5370,0,116,None
7,8885652481,2026-07-07T15:12:21Z,2011,36,23,Turbo,0,unranked,Unranked,True,15,11,14,1747,2869,78860,3948,8289,240,None
8,8885555097,2026-07-07T14:14:07Z,1401,42,23,Turbo,0,unranked,Unranked,True,4,5,7,1111,1810,24159,4130,0,170,None
9,8885481833,2026-07-07T13:29:56Z,1421,8,23,Turbo,0,unranked,Unranked,True,6,4,8,1536,2725,20509,12503,4709,209,None


### Player heroes

,heroId,games,wins,losses,winRate,avgKills,avgDeaths,avgAssists,kda,avgGpm,avgXpm,recentForm
0,60,7,5,2,71.4,7.9,7.7,17.0,3.22,521.7,765.9,"[L, W, W, W, W]"
1,84,5,3,2,60.0,7.8,7.8,13.8,2.77,1002.2,1726.6,"[W, W, L, L, W]"
2,2,4,2,2,50.0,8.2,9.8,11.5,2.03,967.2,1657.8,"[W, L, L, W]"
3,8,3,2,1,66.7,5.3,4.3,6.0,2.62,1342.0,2307.3,"[W, L, W]"
4,3,2,1,1,50.0,2.5,7.5,13.0,2.07,691.0,1822.0,"[L, W]"
5,99,2,1,1,50.0,5.0,7.0,13.0,2.57,845.0,1182.0,"[W, L]"
6,36,2,1,1,50.0,12.0,10.0,8.5,2.05,1156.5,1857.5,"[W, L]"
7,31,2,1,1,50.0,6.0,12.0,16.0,1.83,659.5,1366.5,"[W, L]"
8,75,2,1,1,50.0,7.0,9.0,16.5,2.61,1015.5,2374.0,"[W, L]"
9,135,2,1,1,50.0,6.5,11.0,16.5,2.09,519.0,721.0,"[L, W]"


## Style Score Over 100 Matches

In [8]:
status, style = api_request("GET", f"/v1/players/{ACCOUNT_ID}/style", params={"limit": 100}, allow_error=True)

if not isinstance(style, dict) or style.get("detail"):
    show(f"GET /v1/players/{ACCOUNT_ID}/style?limit=100 -> {status}", style)
    display(Markdown("Style score needs saved player matches. Complete a successful refresh first."))
else:
    show(f"GET /v1/players/{ACCOUNT_ID}/style?limit=100 -> {status}", {key: value for key, value in style.items() if key != "matches"})
    style_matches = pd.DataFrame(style.get("matches") or [])
    for column in ["score", "itemScore", "skillScore"]:
        style_matches[column] = pd.to_numeric(style_matches[column], errors="coerce")
    rated = style_matches.dropna(subset=["score"]).copy()
    by_hero = (
        rated.groupby(["heroId", "heroName"], dropna=False, as_index=False)
        .agg(matches=("matchId", "count"), averageStyle=("score", "mean"), averageItems=("itemScore", "mean"), averageSkills=("skillScore", "mean"))
        .sort_values(["averageStyle", "matches"], ascending=[False, False])
    )
    display(Markdown(f"### Rated matches: {len(rated)} / {len(style_matches)}; average: {rated['score'].mean():.1f}"))
    display(by_hero.round(1))
    display(Markdown("### Most deviant matches"))
    display(rated.sort_values("score", ascending=False)[["matchId", "heroName", "score", "itemScore", "skillScore"]].head(20))

## GET /v1/players/175966938/style?limit=100 -> 200

<IPython.core.display.JSON object>

### Rated matches: 50 / 50; average: 68.5

,heroId,heroName,matches,averageStyle,averageItems,averageSkills
2,3,Bane,2,90.4,100.0,72.7
5,25,Lina,1,90.4,100.0,72.7
13,52,Leshrac,1,90.4,100.0,72.7
17,63,Weaver,1,90.4,100.0,72.7
26,123,Hoodwink,1,90.4,100.0,72.7
3,8,Juggernaut,3,80.1,84.1,72.7
22,99,Bristleback,2,79.6,83.4,72.7
4,14,Pudge,1,79.6,83.3,72.7
19,75,Silencer,2,78.8,82.1,72.7
7,31,Lich,2,75.8,76.8,73.8


### Most deviant matches

,matchId,heroName,score,itemScore,skillScore
15,8883758911,Lina,90.4,100.0,72.7
1,8888227610,Bane,90.4,100.0,72.7
5,8885808627,Weaver,90.4,100.0,72.7
24,8879679595,Bane,90.4,100.0,72.7
21,8883006968,Leshrac,90.4,100.0,72.7
20,8883070149,Lich,90.4,100.0,72.7
19,8883116792,Juggernaut,90.4,100.0,72.7
16,8883728962,Hoodwink,90.4,100.0,72.7
12,8883854478,Ogre Magi,87.3,100.0,63.6
26,8871676285,Axe,81.2,85.7,72.7


## Global Hero Meta

In [10]:
status, hero_winrates = api_request("GET", "/v1/heroes/winrates", params={"limit": 30, "min_matches": 5})
display(pd.DataFrame(hero_winrates).head(30))

status, pudge_build = api_request("GET", "/v1/heroes/14/builds", params={"min_matches": 5, "limit": 5})
show(f"GET /v1/heroes/14/builds -> {status}", pudge_build)

,heroId,name,displayName,shortName,imageUrl,roles,matchCount,winCount,lossCount,winRate
0,67,npc_dota_hero_spectre,Spectre,spectre,None,"[CARRY, ESCAPE, DURABLE]",155204,87256,67948,56.22
1,94,npc_dota_hero_medusa,Medusa,medusa,None,"[CARRY, DURABLE, DISABLER]",49930,27405,22525,54.89
2,12,npc_dota_hero_phantom_lancer,Phantom Lancer,phantom_lancer,None,"[CARRY, ESCAPE, NUKER, PUSHER]",131865,72110,59755,54.68
3,42,npc_dota_hero_skeleton_king,Wraith King,skeleton_king,None,"[CARRY, INITIATOR, DURABLE, DISABLER, SUPPORT]",154556,84066,70490,54.39
4,44,npc_dota_hero_phantom_assassin,Phantom Assassin,phantom_assassin,None,"[CARRY, ESCAPE]",184479,99381,85098,53.87
5,54,npc_dota_hero_life_stealer,Lifestealer,life_stealer,None,"[CARRY, ESCAPE, DURABLE, DISABLER]",126645,67832,58813,53.56
6,113,npc_dota_hero_arc_warden,Arc Warden,arc_warden,None,"[CARRY, ESCAPE, NUKER]",45330,24177,21153,53.34
7,20,npc_dota_hero_vengefulspirit,Vengeful Spirit,vengefulspirit,None,"[ESCAPE, NUKER, INITIATOR, DISABLER, SUPPORT]",176177,93740,82437,53.21
8,41,npc_dota_hero_faceless_void,Faceless Void,faceless_void,None,"[CARRY, ESCAPE, INITIATOR, DURABLE, DISABLER]",139705,74259,65446,53.15
9,80,npc_dota_hero_lone_druid,Lone Druid,lone_druid,None,"[CARRY, DURABLE, PUSHER]",24659,13055,11604,52.94


## GET /v1/heroes/14/builds -> 200

<IPython.core.display.JSON object>